# DPL Confusion Matrix

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from pathlib import Path


## Load Data

In [2]:
# Configuration
dataset = "darpa2000"
scenario = "s1_inside"

In [3]:
logic_files = [
    "darpa", 
    "darpa_neg",
    "darpa_one_net", 
    "darpa_flags", 
    "darpa_flags_neg",
    "darpa_flags_one_net",
]

In [4]:
experiments = {}

for logic_file in logic_files:
    # --- Load metrics ---
    metrics_dir = Path(f"../experiments/{dataset}/{scenario}/deepproblog/{logic_file}/metrics")
    file_paths = list(metrics_dir.iterdir())
        
    for file_path in file_paths:
        experiment_name = f"{file_path.stem[:-16]}" # Remove "run_id"

        # Metrics
        print(f"Processing {experiment_name}...")
        data = np.load(file_path, allow_pickle=True)
        print(experiment_name)
        experiments[experiment_name] = {
            "confusion_matrix": data["confusion_matrix"],
            "classes": data["classes"].tolist(),
            "metrics": data["metrics"].item(),
        }

Processing darpa_scratch_up_w10...
darpa_scratch_up_w10
Processing darpa_pretrained_up_w10...
darpa_pretrained_up_w10
Processing darpa_pretrained_down_w10...
darpa_pretrained_down_w10
Processing darpa_scratch_original_w100...
darpa_scratch_original_w100
Processing darpa_pretrained_up_w100...
darpa_pretrained_up_w100
Processing darpa_pretrained_original_w100...
darpa_pretrained_original_w100
Processing darpa_pretrained_original_w10...
darpa_pretrained_original_w10
Processing darpa_scratch_down_w10...
darpa_scratch_down_w10
Processing darpa_scratch_original_w10...
darpa_scratch_original_w10
Processing darpa_pretrained_down_w50...
darpa_pretrained_down_w50
Processing darpa_pretrained_up_w50...
darpa_pretrained_up_w50
Processing darpa_scratch_up_w50...
darpa_scratch_up_w50
Processing darpa_scratch_down_w50...
darpa_scratch_down_w50
Processing darpa_scratch_original_w50...
darpa_scratch_original_w50
Processing darpa_scratch_down_w100...
darpa_scratch_down_w100
Processing darpa_pretrained_or

## Confusion Matrix Plotting

In [5]:
def compute_masks(cm, classes):
    n = len(classes)
    benign_idx = classes.index("benign")

    diag_mask = np.eye(n, dtype=bool)

    fp_mask = np.zeros_like(cm, dtype=bool)
    fn_mask = np.zeros_like(cm, dtype=bool)

    for i in range(n):
        for j in range(n):
            if j == benign_idx and i != benign_idx:
                fp_mask[i, j] = True   # false alarm
            elif i == benign_idx and j != benign_idx:
                fn_mask[i, j] = True   # missed attack
    
    return diag_mask, fp_mask, fn_mask

In [6]:
def plot_confusion_matrix(
    cm,
    classes,
    experiment_name,
    out_path,
):
    """
    IDS-style confusion matrix visualization.

    Assumes:
        cm[predicted, actual]
    """

    cm_display = np.asarray(cm, dtype=float) + 1

    diag_mask, fp_mask, fn_mask = compute_masks(cm_display, classes)
    masks = {"TP": diag_mask, "FP": fp_mask, "FN": fn_mask}
    cm_colors = {"TP": "Greens", "FP": "Reds", "FN": "Purples"}

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.imshow(cm_display, cmap="Greys", norm=LogNorm(), interpolation="none")

    for label, mask in masks.items():
        ax.imshow(
            np.ma.masked_where(~mask, cm_display),
            cmap=cm_colors[label],
            norm=LogNorm(),
            interpolation="none",
            alpha=0.85,
        )
    
    ax.set_aspect("equal")

    ax.set(
        xticks=np.arange(len(classes)),
        yticks=np.arange(len(classes)),
        xticklabels=classes,
        yticklabels=classes,
        ylabel="Predicted",
        xlabel="Actual",
        title=f"Confusion Matrix - {experiment_name}",
    )

    ax.set_xticks(np.arange(len(classes)+1)-.5, minor=True)
    ax.set_yticks(np.arange(len(classes)+1)-.5, minor=True)
    ax.grid(which="minor", color="lightgray", linestyle='-', linewidth=0.5)
    ax.tick_params(which="minor", bottom=False, left=False)

    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

    # -------------------------
    # Annotate cells
    # -------------------------
    thresh = 20 # cm_display.max() / 10 # np.median(cm_display)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):

            val = cm_display[i, j]
            count = int(cm[i, j])

            # Decide label
            label = ""
            if diag_mask[i, j]:
                label = "TP"
            elif fp_mask[i, j]:
                label = "FP"
            elif fn_mask[i, j]:
                label = "FN"

            text_color = "white" if val > thresh else "black"

            ax.text(
                j,
                i,
                f"{count}\n{label}" if label else f"{count}",
                ha="center",
                va="center",
                color=text_color,
                fontsize=9,
                # fontweight="bold" if label else "normal",
            )

    fig.tight_layout()

    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

    print("Saved confusion matrix plot to:", out_path)

    # plt.show()

In [7]:
for experiment_name, data in experiments.items():
    mat = data["confusion_matrix"]
    classes = data["classes"]

    cm_dir = Path(f"../reports/baselines/{logic_file}/cm")
    cm_dir.mkdir(parents=True, exist_ok=True)
    out_path = f"{cm_dir}/{experiment_name}_cm.png"
    plot_confusion_matrix(mat, classes, experiment_name, out_path)

Saved confusion matrix plot to: ../reports/baselines/darpa_flags_one_net/cm/darpa_scratch_up_w10_cm.png
Saved confusion matrix plot to: ../reports/baselines/darpa_flags_one_net/cm/darpa_pretrained_up_w10_cm.png
Saved confusion matrix plot to: ../reports/baselines/darpa_flags_one_net/cm/darpa_pretrained_down_w10_cm.png
Saved confusion matrix plot to: ../reports/baselines/darpa_flags_one_net/cm/darpa_scratch_original_w100_cm.png
Saved confusion matrix plot to: ../reports/baselines/darpa_flags_one_net/cm/darpa_pretrained_up_w100_cm.png
Saved confusion matrix plot to: ../reports/baselines/darpa_flags_one_net/cm/darpa_pretrained_original_w100_cm.png
Saved confusion matrix plot to: ../reports/baselines/darpa_flags_one_net/cm/darpa_pretrained_original_w10_cm.png
Saved confusion matrix plot to: ../reports/baselines/darpa_flags_one_net/cm/darpa_scratch_down_w10_cm.png
Saved confusion matrix plot to: ../reports/baselines/darpa_flags_one_net/cm/darpa_scratch_original_w10_cm.png
Saved confusion ma